# Módulo 42: Consultas avanzadas en SQL

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m42-consultas-avanzadas-sql-solucion.ipynb](m42-consultas-avanzadas-sql-solucion.ipynb). No es una tarea entregada del programa.

Con los datos ya importados en SQLite, el siguiente paso es **resumirlos y ordenarlos** para responder preguntas de negocio: ¿qué departamentos tienen mejor evaluación?, ¿cómo cambia según el nivel salarial?, ¿cuáles superan un umbral? Trabajamos con el mismo archivo de recursos humanos y llevamos cada resultado a pandas con `read_sql_query`.

1. De SQL a pandas con `read_sql_query`
2. `ORDER BY` con varias claves
3. `GROUP BY` con una columna
4. `GROUP BY` con dos columnas
5. `HAVING`: filtrar grupos
6. Alias en `HAVING`: SQLite frente a T-SQL
7. `ROUND` y la división entera
8. Errores frecuentes
9. Ejercicio

In [1]:
import sqlite3

import pandas as pd

datos = pd.read_csv("recursos-humanos.csv").rename(columns={
    "average_montly_hours": "average_monthly_hours",
    "Work_accident": "work_accident",
    "sales": "department",
    "left": "has_left",
})

conn = sqlite3.connect(":memory:")
datos.to_sql("Empleado", conn, index=False)
print(len(datos), "filas importadas")

14999 filas importadas


Las columnas se renombran al importar, como se explicó en el Módulo 41: `sales` pasa a `department` y `left`, que es palabra reservada, a `has_left`.

## 1. De SQL a pandas con `read_sql_query`

`pd.read_sql_query(sql, conn, params=...)` ejecuta la consulta y devuelve un DataFrame con los nombres de columna del `SELECT`. Así se combina lo mejor de los dos mundos: SQL para filtrar y agregar dentro de la base, pandas para seguir analizando o graficar.

Los valores variables se pasan en `params`, igual que con `execute`.

In [2]:
def q(sql, params=None):
    """Ejecuta una consulta y devuelve el resultado como DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)


q("SELECT department, salary, satisfaction_level FROM Empleado WHERE number_project = ? LIMIT 5", (7,))

,department,salary,satisfaction_level
0,sales,medium,0.11
1,technical,low,0.10
2,support,low,0.10
3,support,low,0.11
4,IT,medium,0.11


## 2. `ORDER BY` con varias claves

Con varias columnas en `ORDER BY`, la primera manda y las siguientes solo **desempatan**. Cada clave lleva su propia dirección: `ORDER BY a ASC, b DESC` ordena por `a` de menor a mayor y, dentro de cada valor de `a`, por `b` de mayor a menor.

En este conjunto hay muchos empates (la satisfacción tiene solo dos decimales), así que las claves de desempate cambian bastante el resultado.

In [3]:
q("""
SELECT number_project, satisfaction_level, last_evaluation
FROM Empleado
ORDER BY number_project ASC, satisfaction_level DESC, last_evaluation DESC
LIMIT 6
""")

,number_project,satisfaction_level,last_evaluation
0,2,1.0,0.86
1,2,1.0,0.86
2,2,1.0,0.74
3,2,1.0,0.74
4,2,1.0,0.63
5,2,1.0,0.39


Para un listado estable conviene terminar con una clave que no se repita. Esta tabla no tiene identificador, pero SQLite ofrece la columna oculta `rowid`, que sí es única.

## 3. `GROUP BY` con una columna

Agrupar por una columna produce una fila por cada valor distinto. Es el equivalente de `df.groupby(col).agg(...)` en pandas.

In [4]:
q("""
SELECT department,
       COUNT(*)                        AS empleados,
       ROUND(AVG(last_evaluation), 3)  AS evaluacion_media
FROM Empleado
GROUP BY department
ORDER BY evaluacion_media DESC
""")

,department,empleados,evaluacion_media
0,management,630,0.724
1,support,2229,0.723
2,technical,2720,0.721
3,accounting,767,0.718
4,IT,1227,0.717
5,marketing,858,0.716
6,product_mng,902,0.715
7,RandD,787,0.712
8,sales,4140,0.710
9,hr,739,0.709


## 4. `GROUP BY` con dos columnas

Con dos columnas, cada grupo es una **combinación** de valores: departamento y nivel salarial. El resultado está en formato "largo" (una fila por combinación); para leerlo como tabla de doble entrada se puede pivotar en pandas.

In [5]:
largo = q("""
SELECT department, salary, ROUND(AVG(last_evaluation), 3) AS evaluacion_media
FROM Empleado
GROUP BY department, salary
ORDER BY department, salary
""")
print(len(largo), "combinaciones")
largo.head(6)

30 combinaciones


,department,salary,evaluacion_media
0,IT,high,0.717
1,IT,low,0.716
2,IT,medium,0.718
3,RandD,high,0.701
4,RandD,low,0.714
5,RandD,medium,0.712


In [6]:
# Formato ancho: una fila por departamento, una columna por nivel salarial
largo.pivot(index="department", columns="salary", values="evaluacion_media")[["low", "medium", "high"]]

salary,low,medium,high
department,,,
IT,0.716,0.718,0.717
RandD,0.714,0.712,0.701
accounting,0.714,0.720,0.725
hr,0.718,0.696,0.744
management,0.713,0.741,0.716
marketing,0.728,0.714,0.664
product_mng,0.726,0.710,0.666
sales,0.709,0.712,0.700
support,0.719,0.729,0.714


## 5. `HAVING`: filtrar grupos

`WHERE` filtra filas antes de agrupar y `HAVING` filtra grupos después. Un uso típico es descartar grupos con pocos datos, cuyos promedios son poco fiables, o quedarse con los grupos que superan un umbral.

In [7]:
q("""
SELECT department,
       COUNT(*)                         AS empleados,
       ROUND(AVG(work_accident), 3)     AS tasa_accidentes
FROM Empleado
WHERE salary <> 'high'
GROUP BY department
HAVING AVG(work_accident) > 0.15
ORDER BY tasa_accidentes DESC
""")

,department,empleados,tasa_accidentes
0,RandD,736,0.170
1,management,405,0.165
2,marketing,778,0.161
3,support,2088,0.150


## 6. Alias en `HAVING`: SQLite frente a T-SQL

Por el orden lógico de ejecución (FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY), un alias definido en el `SELECT` todavía no existe cuando se evalúa `HAVING`. SQL Server sigue esa regla de forma estricta; **SQLite es permisivo** y acepta el alias.

| Cláusula | ¿Acepta el alias del `SELECT`? SQLite | T-SQL |
| --- | --- | --- |
| `WHERE` | Sí (extensión) | No |
| `HAVING` | Sí (extensión) | No |
| `ORDER BY` | Sí | Sí |

In [8]:
# Funciona en SQLite, pero no en SQL Server
q("""
SELECT department, SUM(work_accident) AS accidentes
FROM Empleado
GROUP BY department
HAVING accidentes > 200
ORDER BY accidentes DESC
""")

,department,accidentes
0,sales,587
1,technical,381
2,support,345


La versión portable repite la expresión en `HAVING`, y funciona en ambos motores:

```sql
SELECT department, SUM(work_accident) AS accidentes
FROM Empleado
GROUP BY department
HAVING SUM(work_accident) > 200
ORDER BY accidentes DESC;
```

## 7. `ROUND` y la división entera

`ROUND(x, n)` redondea a `n` decimales. Dos detalles:

- Se redondea **al final**: `ROUND(AVG(x), 2)` no es lo mismo que `AVG(ROUND(x, 2))`, porque redondear cada valor antes de promediar acumula error.
- En SQLite y en SQL Server, la división entre dos enteros es **entera**: `7 / 2` da 3. Para un porcentaje se multiplica antes por `100.0`, que es real.

$$\text{porcentaje} = \frac{100.0 \times \text{casos}}{\text{total}}$$

In [9]:
q("""
SELECT salary,
       SUM(has_left)                                  AS se_fueron,
       COUNT(*)                                       AS total,
       SUM(has_left) / COUNT(*)                       AS division_entera,
       ROUND(100.0 * SUM(has_left) / COUNT(*), 2)     AS pct_rotacion
FROM Empleado
GROUP BY salary
ORDER BY pct_rotacion DESC
""")

,salary,se_fueron,total,division_entera,pct_rotacion
0,low,2172,7316,0,29.69
1,medium,1317,6446,0,20.43
2,high,82,1237,0,6.63


## 8. Errores frecuentes

**Ordenar solo por una clave con muchos empates.** `ORDER BY satisfaction_level DESC LIMIT 5` devuelve 5 de los muchos empleados con satisfacción 1.0, elegidos sin criterio. Si el resultado se usa para algo, se agregan claves de desempate.

In [10]:
q("SELECT COUNT(*) AS con_satisfaccion_maxima FROM Empleado WHERE satisfaction_level = 1.0")

,con_satisfaccion_maxima
0,111


**Llevar a SQL Server una consulta que usa alias en `HAVING` o `WHERE`.** Funciona en SQLite y falla en T-SQL con *"Invalid column name"*. Si la consulta debe ser portable, se repite la expresión.

**Dividir enteros para un porcentaje.** La columna `division_entera` de la sección 7 vale 0 en todos los grupos: los porcentajes reales quedan truncados. La solución es `100.0 *` o `CAST(... AS REAL)`.

**Seleccionar columnas sin agrupar.** Con `GROUP BY department`, añadir `salary` al `SELECT` sin agruparla ni agregarla da un valor arbitrario en SQLite y un error en T-SQL, como se vio en el Módulo 38.

**Usar `WHERE` para condiciones sobre agregados.** `WHERE AVG(x) > 0.5` es un error; esa condición va en `HAVING`.

In [11]:
try:
    q("SELECT department FROM Empleado WHERE AVG(satisfaction_level) > 0.62 GROUP BY department")
except Exception as error:
    print("Error:", str(error).split("': ")[-1])

q("""
SELECT department, ROUND(AVG(satisfaction_level), 3) AS satisfaccion
FROM Empleado
GROUP BY department
HAVING AVG(satisfaction_level) > 0.62
ORDER BY satisfaccion DESC
""")

Error: misuse of aggregate: AVG()


,department,satisfaccion
0,management,0.621


## 9. Ejercicio

Con la misma tabla `Empleado`:

1. Lista los 5 empleados con más horas mensuales; en caso de empate, primero los de mayor evaluación.
2. Calcula, por antigüedad (`time_spend_company`), el número de empleados y el porcentaje que se fue, con dos decimales.
3. Agrupa por número de proyectos y nivel salarial, y muestra solo las combinaciones con al menos 300 empleados y su promedio de horas mensuales redondeado a 1 decimal.
4. Escribe la consulta del punto 3 de forma portable a SQL Server (sin alias en `HAVING`) y comprueba que da el mismo resultado.

### Solución

In [12]:
q("""
SELECT average_monthly_hours, last_evaluation, department
FROM Empleado
ORDER BY average_monthly_hours DESC, last_evaluation DESC
LIMIT 5
""")

,average_monthly_hours,last_evaluation,department
0,310,0.97,support
1,310,0.97,accounting
2,310,0.97,accounting
3,310,0.96,support
4,310,0.96,technical


In [13]:
q("""
SELECT time_spend_company                         AS antiguedad,
       COUNT(*)                                   AS empleados,
       ROUND(100.0 * SUM(has_left) / COUNT(*), 2) AS pct_rotacion
FROM Empleado
GROUP BY time_spend_company
ORDER BY antiguedad
""")

,antiguedad,empleados,pct_rotacion
0,2,3244,1.63
1,3,6443,24.62
2,4,2557,34.81
3,5,1473,56.55
4,6,718,29.11
5,7,188,0.00
6,8,162,0.00
7,10,214,0.00


La rotación es máxima a los 5 años de antigüedad y es nula a partir de los 7.

In [14]:
con_alias = q("""
SELECT number_project, salary, COUNT(*) AS empleados,
       ROUND(AVG(average_monthly_hours), 1) AS horas_mes
FROM Empleado
GROUP BY number_project, salary
HAVING empleados >= 300
ORDER BY number_project, salary
""")

portable = q("""
SELECT number_project, salary, COUNT(*) AS empleados,
       ROUND(AVG(average_monthly_hours), 1) AS horas_mes
FROM Empleado
GROUP BY number_project, salary
HAVING COUNT(*) >= 300
ORDER BY number_project, salary
""")

print("Mismo resultado:", con_alias.equals(portable))
portable

Mismo resultado:

 True


,number_project,salary,empleados,horas_mes
0,2,low,1344,159.1
1,2,medium,904,160.8
2,3,high,408,201.4
3,3,low,1791,196.8
4,3,medium,1856,197.3
5,4,high,368,203.4
6,4,low,2087,206.1
7,4,medium,1910,204.4
8,5,low,1317,212.9
9,5,medium,1199,212.2


In [15]:
conn.close()

## Resumen

- `read_sql_query` lleva el resultado de una consulta a un DataFrame, con parámetros en `params`.
- En `ORDER BY` con varias claves, cada una desempata a la anterior y tiene su propia dirección (`ASC`/`DESC`).
- `GROUP BY` con dos columnas forma un grupo por combinación; el resultado largo se puede pivotar en pandas.
- `HAVING` filtra grupos; SQLite acepta alias en `HAVING` y `WHERE`, pero SQL Server no, así que la forma portable repite la expresión.
- `ROUND` se aplica al final, y para porcentajes se multiplica por `100.0` para evitar la división entera.

En el **Módulo 43** se diseñan tablas relacionadas en SQLite y se consultan con `JOIN`, `CASE` y subconsultas.